This notebook implements the data processing pipeline for the **LLM Semantic Explorer**. 

The core objective of this pipeline is to analyze a Large Language Model's latent space to visually demonstrate how different prompting techniques and their corresponding generated responses are positioned within the semantic space.

The system generates text extracting the hidden state trajectories for each token, performs semantic clustering on them, and projects everything into a three-dimensional space. 
All generated data is aggregated and exported to a JSON file.

These results can then be fully explored in an interactive 3D visualization tool, accessible at [llm-semantic-explorer.vercel.app](https://llm-semantic-explorer.vercel.app)


# Setup and Configuration
In this initial phase, we import the required libraries and define the fundamental parameters for the project. 

We configure the language model to be used (e.g., Llama-3.2-3B-Instruct optimized via Unsloth) and set the text generation constraints. 

This section also defines the essential parameters for the UMAP and HDBSCAN algorithms, which will govern the clustering and spatial projection logic later in the pipeline. 

We also establish the system prompts for generation and clusters labeling.

In [1]:
!pip install unsloth umap-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from kaggle_secrets import UserSecretsClient
from nltk.corpus import stopwords
from pathlib import Path
from string import punctuation
from unsloth import FastLanguageModel
import copy
import hdbscan
import json
import logging
import nltk
import numpy as np
import os
import re
import torch
import umap
import warnings

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


2026-03-08 22:34:56.346616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773009296.720204      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773009296.828736      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773009297.827737      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773009297.827772      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773009297.827775      24 computation_placer.cc:177] computation placer alr

In [3]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

GENERATION_MAX_TOKENS = 256
GENERATION_TEMPERATURE = 0.7

UMAP_CLUSTERING_COMPONENTS = 30
UMAP_3D_COMPONENTS = 3
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_METRIC = "cosine"
UMAP_RANDOM_STATE = 42

HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 2

CLUSTER_LABEL_MAX_TOKENS = 10
CLUSTER_LABEL_TEMPERATURE = 0.1

STOP_WORDS = set(stopwords.words("english"))
PUNCT_SET = set(punctuation)

GENERAION_SYSTEM_PROMPT = (
    "You are a highly analytical and deterministic assistant designed to "
    "generate strictly necessary tokens.\n"
    "Your primary goal is to completing the requested task.\n\n"
    "### RULES:\n"
    "-  **Zero Fluff**: Never use conversational fillers, introductions "
    '(e.g., "Here is the answer:", "Sure!"), or conclusions '
    '(e.g., "I hope this helps!"). Do not greet the user.\n'
    "-  **Brevity**: Keep sentences as short and direct as possible.\n"
    "-  **Formatting Constraints**: Output plain text. Do not use markdown "
    "formatting."
)

CLUSTER_LABELING_SYSTEM_PROMPT = (
    "You are an expert computational linguist and data scientist. Your task "
    "is to analyze a list of text tokens that have been grouped into a single "
    "cluster based on their semantic similarity or syntactic role. You must "
    "provide a highly descriptive label that captures the core theme, concept, "
    "or grammatical function of the group.\n\n"
    "CRITICAL CONSTRAINTS:\n"
    "- Output EXACTLY 1 to 5 words.\n"
    "- Output ONLY the label itself. Never include preambles like "
    "'The label is' or 'Label:'.\n"
    "- Do NOT use quotes, periods, or any other punctuation.\n"
    "- Use Title Case."
)

logging.raiseExceptions = False
warnings.filterwarnings('ignore')

# Pipeline Helper Functions

## 1. LLM Loading and Initialization
Here, we initialize the language model and its corresponding tokenizer. 

To optimize computational resources and GPU memory footprint without sacrificing significant semantic accuracy, the model is loaded in a 4-bit quantized format.

This approach is an industry standard for efficient local inference, allowing us to run complex models even on constrained hardware while maintaining high-quality hidden state representations.

In [4]:
def load_model(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=LOAD_IN_4BIT):
    """Load and prepare the language model for inference."""
    token = UserSecretsClient().get_secret("HF_TOKEN")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        token=token,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer

## 2. Text Generation and Hidden States Extraction
This module is the core of the inference pipeline: for each prompt provided, the model sequentially generates response tokens. 

In parallel with the standard generation process, we execute a specialized forward pass to capture the **hidden state vectors** from the neural network's final layer for every generated token. 

This sequence of high-dimensional vectors effectively creates a **trajectory** that maps the logical and semantic path the model took to formulate its response.

In [5]:
def generate_with_path(user_prompt, model, tokenizer):
    """
    Generate a response and capture hidden-state vectors for each token.

    Returns a dict with keys: prompt, response, tokens, path.
    """
    messages = [
        {"role": "system", "content": GENERAION_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=GENERATION_MAX_TOKENS,
            use_cache=True,
            temperature=GENERATION_TEMPERATURE,
            return_dict_in_generate=False,
        )

        generated_ids = outputs[0][prompt_length:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        generated_tokens = [
            tokenizer.decode(t, skip_special_tokens=True) for t in generated_ids
        ]

        # Single forward pass to get clean hidden states
        forward_outputs = model(outputs, output_hidden_states=True)
        last_layer = forward_outputs.hidden_states[-1]

        path_tensor = last_layer[0, prompt_length:, :]
        path = path_tensor.cpu().float().numpy()

    return {
        "prompt": user_prompt,
        "response": generated_text,
        "tokens": generated_tokens,
        "path": path,
    }

In [6]:
def process_prompts(prompts_config, model, tokenizer):
    """Process prompts respecting inter-prompt dependencies."""
    results = {}

    for name, config in prompts_config.items():
        print(f"\n--- Generating response for {name} ---")
        
        if isinstance(config, str):
            prompt_text = config
            deps = []
        else:
            prompt_text = config.get("prompt", "")
            deps = config.get("deps", [])

        print(prompt_text)
        print("\n---\n")
        
        parts = []
        for dep in deps:
            if dep in results:
                parts.append(results[dep]["response"])
                
        parts.append(prompt_text)
        prompt = "\n\n".join(parts)
        
        results[name] = generate_with_path(prompt, model, tokenizer)

        print(f"{results[name]['response']}")
        print("\n---\n")
        
        print(f"Generated tokens: {len(results[name]['tokens'])}")
        
    return results

## 3. Relevant Token Filtering
To obtain a clean semantic map and reduce visual clutter in our final 3D representation, the generated tokens must pass through a filtering mechanism. 

We systematically remove purely syntactic tokens, standard stop words (like "and", "the", "is"), isolated punctuation marks, and model-specific special tags (such as end-of-sequence markers). 

By discarding this structural noise, we ensure that we preserve and visualize only the tokens that carry genuine semantic meaning and contribute to the core concept of the response.

In [7]:
def is_relevant(token):
    """Return True if the token is semantically meaningful."""
    token = token.strip()
    if not token:
        return False
    if re.match(r"^<\|.*?\|>$", token):
        return False
    if all(ch in PUNCT_SET for ch in token):
        return False
    if token.lower() in STOP_WORDS:
        return False
    return True

In [8]:
def extract_relevant_tokens(results):
    """
    Filter tokens to keep only semantically relevant ones.

    Returns:
        all_vectors       – every token vector (relevant or not)
        relevant_vectors  – only the relevant token vectors
        relevant_tokens   – matching token strings
        relevant_steps    – original step indices
        path_indices      – {name: (start, end)} index ranges per prompt
    """
    all_vectors = []
    
    relevant_vectors = []
    relevant_tokens = []
    relevant_steps = []
    path_indices = {}

    for name, data in results.items():
        start_idx = len(relevant_vectors)

        for step_idx, (tok, vec) in enumerate(zip(data["tokens"], data["path"])):
            all_vectors.append(vec)
            if is_relevant(tok):
                relevant_vectors.append(vec)
                relevant_tokens.append(tok)
                relevant_steps.append(step_idx)

        path_indices[name] = (start_idx, len(relevant_vectors))

    return (
        np.array(all_vectors),
        np.array(relevant_vectors),
        relevant_tokens,
        relevant_steps,
        path_indices,
    )

## 4. Semantic Clustering (UMAP + HDBSCAN)
Clustering raw LLM hidden states directly is highly ineffective due to the "curse of dimensionality" in high-dimensional spaces. 

To solve this, we rely on the powerful synergy of **UMAP** and **HDBSCAN**:

* **UMAP (Dimensionality Reduction):** We first compress the vectors into a manageable intermediate space. Unlike PCA or t-SNE, UMAP is exceptionally good at preserving both the local neighborhoods and the global topological structure of the semantic space.
* **HDBSCAN (Density-Based Clustering):** We then apply HDBSCAN to find natural groupings. Unlike K-Means, HDBSCAN does not require guessing the number of clusters in advance. Furthermore, it doesn't force every token into a group, effectively isolating ambiguous or transitionary tokens as "noise" (label -1).

UMAP untangles and densifies the semantic space, allowing HDBSCAN to accurately extract naturally occurring pockets of meaning while safely ignoring irrelevant outliers.

In [9]:
def cluster_tokens(vectors):
    """
    Reduce dimensionality with UMAP, then cluster with HDBSCAN.
    
    Returns an array of cluster labels (–1 = noise).
    """
    N = vectors.shape[0]
    
    # Ensure components and neighbors are valid for the dataset size
    safe_components = max(1, min(UMAP_CLUSTERING_COMPONENTS, N - 2))
    safe_neighbors = max(2, min(UMAP_N_NEIGHBORS, N - 1))
    
    reduced = umap.UMAP(
        n_neighbors=safe_neighbors,
        n_components=safe_components,
        metric=UMAP_METRIC,
        random_state=UMAP_RANDOM_STATE,
    ).fit_transform(vectors)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(reduced)

    n_clusters = len(set(labels) - {-1})
    print(f"Clusters found: {n_clusters}")
    print(f"Noise: {int(np.sum(labels == -1))} tokens")
    # for c in np.unique(labels):
    #     count = int(np.sum(labels == c))
    #     tag = "Noise" if c == -1 else f"Cluster_{c}"
    #     print(f"  {tag}: {count} tokens")

    return labels

In [10]:
def collect_cluster_tokens(tokens, labels):
    """Group unique tokens by cluster label, ignoring noise (–1)."""
    cluster_map = {}
    for token, label in zip(tokens, labels):
        if label == -1:
            continue
        cluster_map.setdefault(int(label), set()).add(token)
    return {k: list(v) for k, v in cluster_map.items()}

## 5. LLM-Powered Cluster Labeling
Once the tokens are grouped, we employ the LLM itself as an automated "computational linguist" to assign a meaningful name to each cluster. 

By feeding the model the list of tokens belonging to a specific cluster, we prompt it to generate a highly concise label that accurately captures the central theme, overarching concept, or grammatical function of that specific group. 

This step provides human-readable context to the mathematical clusters.

In [11]:
def label_clusters(cluster_tokens_map, model, tokenizer):
    """Use the LLM to generate a short descriptive label for each cluster."""
    cluster_names = {}

    for cluster_id, tokens in cluster_tokens_map.items():
        tokens_str = ", ".join(f"'{t}'" for t in tokens)
        user_prompt = (
            f"Cluster tokens:\n[{tokens_str}]\n\n"
            "Provide the best 1-3 word label for this cluster. "
            "Remember: output ONLY the exact label text."
        )

        messages = [
            {"role": "system", "content": CLUSTER_LABELING_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        prompt_length = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=CLUSTER_LABEL_MAX_TOKENS,
                temperature=CLUSTER_LABEL_TEMPERATURE,
            )

        generated_ids = outputs[0][prompt_length:]
        label = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        cluster_names[cluster_id] = label
        print(f"Cluster {cluster_id}: {label}")

    return cluster_names

## 6. 3D Spatial Projection
The filtered semantic vectors are now drastically reduced to exactly three physical dimensions (x, y, and z coordinates) using the **UMAP** algorithm once again. 

While compressing thousands of dimensions into just three inherently results in some information loss, UMAP is designed to preserve local topographies and relative distances as much as possible. 

This extreme projection translates the abstract vocabulary space into a format compatible with the 3D visual representation required for our front-end application.

In [12]:
def project_to_3d(vectors):
    """Project token vectors to 3 dimensions via UMAP."""
    reducer = umap.UMAP(
        n_components=UMAP_3D_COMPONENTS,
        metric=UMAP_METRIC,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        random_state=UMAP_RANDOM_STATE,
    )
    embeddings = reducer.fit_transform(vectors)
    # print(f"3D embeddings shape: {embeddings.shape}")
    return embeddings

## 7. Data Assembly and JSON Export
The final processing step formats all the extracted and computed data into an optimized JSON structure. 

This structure includes the calculated 3D spatial coordinates, the raw text of the tokens, the tracking paths of the different prompt chains, and the LLM-generated cluster labels. 

The resulting file acts as the complete vector Knowledge Base, serving as the foundational dataset that will power the interactive 3D visualization front-end described in the introduction.

In [13]:
def build_output(results, path_indices, embeddings_3d, relevant_tokens, relevant_steps, cluster_labels, cluster_names, relevant_vectors):
    """Assemble the final JSON-serializable output dict."""
    paths = []

    for path_id, (name, (start_idx, end_idx)) in enumerate(path_indices.items()):
        points = [
            {
                "position": embeddings_3d[j].tolist(),
                "raw": relevant_vectors[j].tolist(),
                "token": relevant_tokens[j],
                "step": int(relevant_steps[j]),
                "path_id": path_id,
                "cluster_id": int(cluster_labels[j]),
            }
            for j in range(start_idx, end_idx)
        ]
        paths.append({
            "id": path_id,
            "name": name,
            "prompt": results[name]["prompt"],
            "response": results[name]["response"],
            "points": points,
        })

    clusters = [
        {
            "id": int(cid),
            "name": cname,
            "count": int(np.sum(cluster_labels == cid)),
        }
        for cid, cname in cluster_names.items()
    ]

    return {"paths": paths, "clusters": clusters}

In [14]:
def save_output(output, filename):
    """Write the output dicts to out/3d and out/full directories."""
    os.makedirs("out/3d", exist_ok=True)
    os.makedirs("out/full", exist_ok=True)
    
    out_full_filepath = os.path.join("out/full", f"{filename}.json")
    with open(out_full_filepath, "w") as f:
        json.dump(output, f, indent=2)

    print(f"Saved full data to {out_full_filepath}")
    
    output_3d = copy.deepcopy(output)
    for path in output_3d["paths"]:
        for point in path["points"]:
            if "raw" in point:
                del point["raw"]
                
    out_3d_filepath = os.path.join("out/3d", f"{filename}.json")
    with open(out_3d_filepath, "w") as f:
        json.dump(output_3d, f, indent=2)
        
    print(f"Saved 3d data to {out_3d_filepath}")

In [15]:
def print_summary(output):
    """Print a human-readable summary of the generated data."""
    for path in output["paths"]:
        pid = path["points"][0]["path_id"] if path["points"] else "N/A"
        print(f"  Path '{path['name']}' (ID: {pid}): {len(path['points'])} points")
    print(f"  Clusters: {output['clusters']}")

# Main Pipeline Execution
Finally, we trigger the entire processing chain previously defined.

In [16]:
def load_inputs():
    input_dir = '/kaggle/input/datasets/dammafra/llm-semantic-explorer-dataset'
    inputs = {}
    
    for dirname, _, filenames in os.walk(input_dir):
        for filename in filenames:
            if filename.endswith('.json'):
                file_path = os.path.join(dirname, filename)
                with open(file_path, 'r', encoding='utf-8') as file:
                    inputs[Path(filename).stem] = json.load(file)

    return inputs

In [17]:
def pipeline(model, tokenizer, filename, prompts_config):
    print(f"\n=== Processing {filename} ===")
    results = process_prompts(prompts_config, model, tokenizer)

    print("\n--- Filtering tokens ---")
    all_vecs, rel_vecs, rel_tokens, rel_steps, path_idx = extract_relevant_tokens(results)
    print(f"Total tokens: {all_vecs.shape[0]}")
    print(f"Relevant tokens: {rel_vecs.shape[0]}")

    print("\n--- Clustering ---")
    labels = cluster_tokens(rel_vecs)

    print("\n--- Labeling clusters ---")
    cluster_tok_map = collect_cluster_tokens(rel_tokens, labels)
    cluster_names = label_clusters(cluster_tok_map, model, tokenizer)

    print("\n--- 3D projection ---")
    embeddings_3d = project_to_3d(rel_vecs)

    print("\n--- Building output ---")
    output = build_output(
        results, 
        path_idx, 
        embeddings_3d, 
        rel_tokens, 
        rel_steps, 
        labels, 
        cluster_names,
        rel_vecs
    )
    
    save_output(output, filename)
    # print_summary(output)
    print("\n")
    
    return output

In [18]:
def main():
    nltk.download("stopwords", quiet=True)

    print("Loading inputs...")
    inputs = load_inputs()
                
    print("Loading model...")
    model, tokenizer = load_model()

    for technique, config in inputs.items():
        pipeline(model, tokenizer,technique, config)
    
main()

Loading inputs...
Loading model...
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.



=== Processing chain-of-thought ===

--- Generating response for bad_prompt ---
Five friends (Alice, Bob, Charlie, David, Eve) are sitting in a row at a movie theater. Bob is sitting immediately to the left of David. Eve is sitting at one of the ends. Charlie is sitting somewhere to the right of Alice but to the left of Bob. Who is sitting in the middle seat?

---

Alice is sitting in the middle seat.

---

Generated tokens: 9

--- Generating response for good_prompt ---
Five friends (Alice, Bob, Charlie, David, Eve) are sitting in a row at a movie theater. Bob is sitting immediately to the left of David. Eve is sitting at one of the ends. Charlie is sitting somewhere to the right of Alice but to the left of Bob. Who is sitting in the middle seat? Let's think step by step, explicitly deducing the position of each person one by one before giving the final answer.

---

Let's break down the problem step by step.

1. Eve is sitting at one of the ends, so she can only occupy either the le

In [19]:
import shutil
import subprocess

def push_datasets_to_github():
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_name = user_secrets.get_secret("GITHUB_REPO_NAME")
    repo_owner = user_secrets.get_secret("GITHUB_REPO_OWNER")
        
    source_dir = "/kaggle/working/out/3d"
    repo_dir = f"/kaggle/working/{repo_name}"
    dest_dir = os.path.join(repo_dir, "public/datasets")    
    repo_url = f"https://{github_token}@github.com/{repo_owner}/{repo_name}.git"
    
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull"], check=True)
        
    if not os.path.exists(source_dir):
        return
        
    os.makedirs(dest_dir, exist_ok=True)
    
    for filename in os.listdir(source_dir):
        src_path = os.path.join(source_dir, filename)
        dest_path = os.path.join(dest_dir, filename)
        if os.path.isfile(src_path):
            if os.path.exists(dest_path):
                os.remove(dest_path)
            shutil.copy2(src_path, dest_path)
            
    subprocess.run(["git", "-C", repo_dir, "config", "user.email", "bot@kaggle.com"], check=True)
    subprocess.run(["git", "-C", repo_dir, "config", "user.name", "Kaggle Auto-Commit"], check=True)
    
    subprocess.run(["git", "-C", repo_dir, "add", "public/datasets/"], check=True)    
    subprocess.run(["git", "-C", repo_dir, "commit", "-m", "Kaggle Datasets Update"], check=True)    
    subprocess.run(["git", "-C", repo_dir, "push"], check=True)

    shutil.rmtree(repo_dir, ignore_errors=True)

push_datasets_to_github()

Cloning into '/kaggle/working/llm-semantic-explorer'...


[main 852b6fc] Kaggle Datasets Update
 6 files changed, 9870 insertions(+), 10752 deletions(-)
 rewrite public/datasets/step-back-prompting.json (60%)


To https://github.com/dammafra/llm-semantic-explorer.git
   29c8aae..852b6fc  main -> main


In [20]:
!rm -rf ./huggingface_tokenizers_cache
!rm -rf ./unsloth_compiled_cache